# Mini encoder-decoder model with self-attention on a toy dataset

## Imports and Setup

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Embedding,
    Dense,
    LayerNormalization
)
import numpy as np

## 1. Data

In [2]:
# Toy dataset (English → French)
data_pairs = [
    ("hello", "bonjour"),
    ("how are you", "comment ça va"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("good night", "bonne nuit"),
]

# Build vocabularies
src_vocab = {"<pad>":0, "<start>":1, "<end>":2, "hello":3, "how":4, "are":5, "you":6, "good":7, "morning":8, "night":9, "thank":10}
tgt_vocab = {"<pad>":0, "<start>":1, "<end>":2, "bonjour":3, "comment":4, "ça":5, "va":6, "merci":7, "bonne":8, "nuit":9}

## 2. Positional Encoding

In [3]:
def positional_encoding(seq_len, d_model):
  pos = np.arange(seq_len)[:, np.newaxis]
  i = np.arange(d_model)[:, np.newaxis]
  angle_rate = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
  angle_rads = pos * angle_rate

  sines = np.sin(angle_rads[:, 0::2])
  cosines = np.cos(angle_rads[:, 1::2])
  pos_encoding = np.concatenate([sines, cosines], axis=1)
  return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

## 3. Scaled dot-product attention

In [4]:
def scaled_dot_product_attention(Q, K, V, mask=None):
  matmul_qk = tf.matmul(Q, K, transpose_b=True)
  dk = tf.cast(tf.shape(K)[-1], tf.float32)
  scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

  if mask is not None:
    scaled_attention_logits += (mask * -1e9)

  attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
  output = tf.matmul(attention_weights, V)
  return output, attention_weights

## 4. Multi-head attention layer

In [5]:
class MultiHeadAttention(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads):
    super().__init__()
    self.num_heads = num_heads
    self.d_model = d_model

    assert d_model % self.num_heads == 0
    self.depth = d_model // self.num_heads

    self.Wq = Dense(d_model)
    self.Wk = Dense(d_model)
    self.Wv = Dense(d_model)
    self.dense = Dense(d_model)

  def split_heads(self, x, batch_size):
    x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
    return tf.transpose(x, perm=[0, 2, 1, 3])

  def call(self, v, k, q, mask):
    batch_size = tf.shape(q)[0]

    Q = self.Wq(q)
    K = self.Wk(k)
    V = self.Wv(v)

    Q = self.split_heads(Q, batch_size)
    K = self.split_heads(K, batch_size)
    V = self.split_heads(V, batch_size)

    scaled_attention, _ = scaled_dot_product_attention(Q, K, V, mask)
    scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
    concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))

    return self.dense(concat_attention)

## 5. Encoder-Decoder blocks

In [7]:
# Encoder layer
class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, dff):
    super().__init__()

    self.mha = MultiHeadAttention(d_model, num_heads)
    self.ffn = tf.keras.Sequential([
        Dense(dff, activation="relu"),
        Dense(d_model)
    ])

    self.layernorm1 = LayerNormalization(epsilon=1e-6)
    self.layernorm2 = LayerNormalization(epsilon=1e-6)

  def call(self, x, mask):
    attn_output = self.mha(x, x, x, mask)
    out1 = self.layernorm1(x + attn_output)

    ffn_output = self.ffn(out1)
    out2 = self.layernorm2(out1 + ffn_output)

    return out2

In [8]:
# Decoder layer
class DecoderLayer(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, dff):
    super().__init__()

    self.mha1 = MultiHeadAttention(d_model, num_heads)
    self.mha2 = MultiHeadAttention(d_model, num_heads)

    self.ffn = tf.keras.Sequential([
        Dense(dff, activation="relu"),
        Dense(d_model)
    ])

    self.layernorm1 = LayerNormalization(epsilon=1e-6)
    self.layernorm2 = LayerNormalization(epsilon=1e-6)
    self.layernorm3 = LayerNormalization(epsilon=1e-6)

  def call(self, x, enc_output, look_ahead_mask, padding_mask):
    attn1 = self.mha1(x, x, x, look_ahead_mask)
    out1 = self.layernorm1(x + attn1)

    attn2 = self.mha2(enc_output, enc_output, out1, padding_mask)
    out2 = self.layernorm2(out1 + attn2)

    ffn_output = self.ffn(out2)
    out3 = self.layernorm3(out2 + ffn_output)

    return out3

## Put it altogether

In [9]:
sample_encoder_layer = EncoderLayer(512, 8, 2048)
sample_decoder_layer = DecoderLayer(512, 8, 2048)

# Example input
x = tf.random.uniform((64, 43, 512))
enc_output = tf.random.uniform((64, 43, 512))

y = tf.random.uniform((64, 5, 512))
dec_output = sample_decoder_layer(y, enc_output, None, None)

# print output shapes
print(enc_output.shape)
print(dec_output.shape)

(64, 43, 512)
(64, 5, 512)
